# FinQA: teach the small model to pick its figures first

Trains Gemma 3 4B on 2,500 solved questions from FinQA's training split. Each answer to learn lists the figures first, then the formula. Then it answers each of the 297 unseen test questions three times, so we can see when it is unsure. About 2.5 to 3 hours.

1. **Runtime > Change runtime type > T4 GPU**, save.
2. **Runtime > Run all**. No uploads.
3. Keep the tab open. `fin_figures_adapter.zip` downloads after training (keep it), and `fin_figures_results.zip` at the end: send that one to Claude.

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b v2-split https://github.com/eshwarpotturi/slm-opti /content/slm-opti
!rm -rf /content/FinQA && git clone -q --depth 1 https://github.com/czyssrs/FinQA /content/FinQA

In [ ]:
import os, sys, json, time, glob, zipfile, random
os.chdir("/content/slm-opti"); sys.path.insert(0, "/content/slm-opti"); sys.path.insert(0, "/content/slm-opti/fin")
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", max_seq_length=3072, load_in_4bit=True, full_finetuning=False)
import stack, run, prep
run.TUNED = run.TUNED_FIG      # the new instruction: figures first, then the formula
run.VOTES = 3                  # answer each test question three times

def _msgs(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

@torch.no_grad()
def hf_chat(messages, temperature=0, max_tokens=400):
    inputs = tokenizer.apply_chat_template(_msgs(messages), add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    kw = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw["temperature"] = temperature
    out = model.generate(**inputs, **kw)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
stack.HF_CHAT = hf_chat

## Training examples: figures first, then the formula

In [ ]:
raw = json.load(open("/content/FinQA/dataset/train.json"))
pool = []
for x in raw:
    t = prep.slim(x)
    if len(run.render(t).split()) > 900 or not t["facts"]:
        continue
    if isinstance(t["answer"], str):
        last = "ANSWER: " + t["answer"]
    elif t["formula"] and run._safe(t):
        last = "FORMULA: " + t["formula"]
    else:
        continue
    target = "Figures:\n" + "\n".join("- " + f.strip()[:220] for f in t["facts"]) + "\n" + last
    pool.append((t, target))
random.Random(7).shuffle(pool)
N_TRAIN = 2500
from datasets import Dataset
rows = []
for t, target in pool[:N_TRAIN]:
    msgs = [{"role": "user", "content": [{"type": "text", "text": run.TUNED + "\n\n" + run.ask(t)}]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]}]
    rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")})
ds = Dataset.from_list(rows)
print(len(pool), "usable training questions;", len(rows), "used")
print(pool[0][1])

In [ ]:
model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", per_device_train_batch_size=1, gradient_accumulation_steps=8, warmup_steps=10,
    num_train_epochs=1, learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
    lr_scheduler_type="linear", seed=3407, report_to="none", output_dir="outputs"))
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
os.makedirs("fin/results", exist_ok=True)
t0 = time.time(); trainer.train()
json.dump({"minutes": round((time.time() - t0) / 60, 1), "examples": len(rows), "log": trainer.state.log_history},
          open("fin/results/finetune_log_figures.json", "w"))
print("Training took %.0f minutes." % ((time.time() - t0) / 60))
model.save_pretrained("/content/fin_figures_adapter")
import shutil
shutil.make_archive("/content/fin_figures_adapter", "zip", "/content/fin_figures_adapter")
from google.colab import files
files.download("/content/fin_figures_adapter.zip")

## The 297 unseen questions, three answers each

In [ ]:
model.eval()
t0 = time.time()
run.main(["hf:gemma-3-4b-figures", "--layers", "24", "--tag", "fig", "--tuned", "--workers", "1", "--budget", "100000"])
print("Testing took %.0f minutes." % ((time.time() - t0) / 60))
rs = [json.loads(l) for l in open("fin/results/gemma-3-4b-figures__L24-fig.jsonl")]
first = sum(r.get("first_right", False) for r in rs)
sure = [r for r in rs if r.get("agree") == 3]
print("First answer correct: %d of %d (%.0f%%)" % (first, len(rs), 100 * first / len(rs)))
print("All three answers agree on %d questions; %d of those are correct" % (len(sure), sum(r["right"] for r in sure)))
with zipfile.ZipFile("/content/fin_figures_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["fin/results/gemma-3-4b-figures__L24-fig.jsonl", "fin/results/finetune_log_figures.json"]:
        z.write(f)
files.download("/content/fin_figures_results.zip")